In [2]:
import sys
import os
import numpy as np
from tqdm import tqdm

sys.path.append(os.path.abspath(".."))

from gymnasium.vector import SyncVectorEnv
from core.env.core import SnakeEnv
from core.env.types import ObsType
from agents.q_learning import QLearningAgent

num_envs, total_episodes = 16, 250000
env = SyncVectorEnv(
    [
        lambda i=i: SnakeEnv(
            width=20,
            height=20,
            obs_type=ObsType.VECTOR_11,
            num_apples=3,
            num_obstacles=15,
            seed=42 + i,
        )
        for i in range(num_envs)
    ]
)

epsilon_decay = (0.01 / 1.0) ** (1 / (total_episodes * 0.7))

agent = QLearningAgent(
    state_dim=2048,
    action_dim=3,
    lr=0.05,
    gamma=0.99,
    epsilon_decay=epsilon_decay,
    seed=42,
)

training_logs, episode_rewards, completed = [], np.zeros(num_envs), 0
obs, infos = env.reset()

best_reward = -np.inf

with tqdm(total=total_episodes, desc="Parallel Training") as pbar:
    while completed < total_episodes:
        actions = [agent.act(o) for o in obs]
        next_obs, rewards, terms, truncs, next_infos = env.step(actions)

        for i, (o, a, r, no, term, trunc) in enumerate(
            zip(obs, actions, rewards, next_obs, terms, truncs)
        ):
            agent.update(o, a, r, no, term)
            episode_rewards[i] += r

            if term or trunc:
                completed += 1
                if completed <= total_episodes:
                    pbar.update(1)
                    agent.train()  # Decay epsilon per episode

                    reward_val = episode_rewards[i]
                    training_logs.append(
                        {
                            "episode": completed,
                            "reward": reward_val,
                            "epsilon": agent.epsilon,
                        }
                    )

                    if len(training_logs) >= 100:
                        recent_avg = np.mean(
                            [log["reward"] for log in training_logs[-100:]]
                        )
                        if recent_avg > best_reward:
                            best_reward = recent_avg
                            agent.save("q_learning_snake_best.pkl")

                    if completed % 5000 == 0:
                        recent_avg = (
                            np.mean([log["reward"] for log in training_logs[-100:]])
                            if len(training_logs) >= 100
                            else reward_val
                        )
                        tqdm.write(
                            f"Ep {completed}/{total_episodes} | Avg Reward (Shaped, last 100): {recent_avg:.2f} | Eps: {agent.epsilon:.3f} | Best Avg: {best_reward:.2f}"
                        )
                episode_rewards[i] = 0

        obs = next_obs
        infos = next_infos

env.close()

Parallel Training:   2%|▏         | 5097/250000 [00:10<08:34, 475.98it/s]

Ep 5000/250000 | Avg Reward (Shaped, last 100): -2.12 | Eps: 0.877 | Best Avg: 2.93


Parallel Training:   4%|▍         | 10070/250000 [00:21<09:00, 443.70it/s]

Ep 10000/250000 | Avg Reward (Shaped, last 100): -1.72 | Eps: 0.769 | Best Avg: 11.12


Parallel Training:   6%|▌         | 15125/250000 [00:33<09:08, 428.56it/s]

...

Ep 240000/250000 | Avg Reward (Shaped, last 100): 726.08 | Eps: 0.010 | Best Avg: 1045.96


Parallel Training:  98%|█████████▊| 245008/250000 [45:29<01:35, 52.49it/s]

Ep 245000/250000 | Avg Reward (Shaped, last 100): 787.05 | Eps: 0.010 | Best Avg: 1045.96


Parallel Training: 100%|██████████| 250000/250000 [47:03<00:00, 88.54it/s]

Ep 250000/250000 | Avg Reward (Shaped, last 100): 737.69 | Eps: 0.010 | Best Avg: 1045.96


In [3]:
agent.save("q_learning_snake.pkl")

In [4]:
from core.utils import save_metrics

save_metrics(training_logs, "q_learning_training_logs.csv")

In [5]:
from core.utils import evaluate_agent

evaluate_agent(agent)

Evaluating Agent: 100%|██████████| 100/100 [00:02<00:00, 43.56it/s]


Metric          | Average  | Max     
-----------------------------------
Rewards         | 897.59   | 2379.50 
Apples          | 18.26    | 48.00   
Steps           | 270.44   | 871.00  

Death Distribution:
 - self: 81 (81.0%)
 - wall: 19 (19.0%)



({'avg': 897.5920000000002, 'max': 2379.500000000008},
 {'avg': 18.26, 'max': 48.0},
 {'avg': 270.44, 'max': 871.0},
 {<DeathReason.SELF: 'self'>: 81, <DeathReason.WALL: 'wall'>: 19})